# PUMP physics-based features

Adapted from `clean_physics_based_features_automated(2).ipynb`. The FAN numerical feature definitions are retained where applicable; PUMP loading, identities, states, and region selection replace the FAN-specific parts.

Run the cells in order. Put the two supplied PUMP CSVs beside the notebook, or edit `META_PATH` and `TIMESTAMPS_PATH`. The raw sensor files must exist at the paths in the metadata; use `PATH_REPLACEMENTS` if the drive/root changed. Dependencies: `numpy pandas polars scipy matplotlib tqdm`.

- Default: healthy + single faults, determined from nonzero fault-code positions. The supplied data selects **315 paired samples / 18 classes**; `INCLUDE_MULTIFAULTS=True` selects all **613 pairs / 38 classes**.
- Use every interval between consecutive signalwise changepoints: **Air has regions 0–1; Water has regions 0–3**. Intervals are `[start, end)`. Supplied indices refer to raw-file rows; rows are never filtered or reordered before slicing.
- One output row per `fault_id@pump_id@sample_id`, with both states joined by ID. Example feature: `CV1_air_0_col_0_rms`. Region 0 means the first interval; it does not label an operating condition by itself.
- Required channels match the supplied PUMP reference: AC1/GYR have three axes; VRY has two columns; CV1, CTB, CTR, SMG, PRS and WTF use column 0. PRS/WTF apply only to Water. No extra channels are silently added.
- The uploaded metadata references **legacy data only**. Legacy `Timestamp` is seconds. A future merged file may use `firmware_timestamp` or `F_Timestamp` in microseconds. New electrical channel equivalents must be explicitly set in `NEW_RAW_COLUMNS`; the older inference mapping is not assumed to prove physical/calibration equivalence.
- Output files are saved in `pump_features`. All four notebooks are independent and use the same sample keys; no inference config, checkpoint or helper file is needed.

The FAN peak/trough-amplitude calculation and last-five-cycle/10%-tolerance stability rule are applied to the first 0.62 seconds of each region. Peak separation is expressed in seconds so differing sensor sampling rates do not change its physical meaning. As in FAN, amplitude pairs >=10000 are rejected; review that threshold if units/calibration change.

`cycle_count` is the number of detected waveform cycles in this short window; **it is not pump revolutions**. `settling_time_s` is relative to that region's first changepoint. If `stability_observed=0`, this duration is only an observation bound. Region starts can be operating transitions rather than motor startup, and the final five cycles are only a local reference.

The RPM-decay section is omitted because the supplied PUMP metadata has no RPM signal. No pressure/flow proxy is substituted for RPM.

In [1]:
import os

# Conservative thread limits to reduce sustained CPU load.
# Restart the kernel before running this notebook.
os.environ["POLARS_MAX_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import gc
import pandas as pd
import numpy as np
import polars as pl
import matplotlib.pyplot as plt

from tqdm import tqdm
from scipy.signal import find_peaks

In [2]:
from pathlib import Path

# Only edit these paths if the CSVs are not beside this notebook.
META_PATH = Path("pump_meta_combined_all_faults_mapped.csv")
TIMESTAMPS_PATH = Path("pump_signalwise_timestamp_combined.csv")
OUTPUT_DIR = Path("pump_features")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
INCLUDE_MULTIFAULTS = False  # Same healthy + single-fault selection as FAN.
STATES = ("air", "water")
REGION_EDGE_SECONDS = 0.25

# Optional relocation of the raw files; leave empty if the CSV paths are valid.
PATH_REPLACEMENTS = {}  # e.g. {r"D:\VGuard Ahemedebad Data": r"E:\PumpData"}

# These are the legacy columns used by the supplied PUMP feature/config files.
SIGNAL_COLUMNS = {
    "AC1": ["0", "1", "2"], "GYR": ["0", "1", "2"],
    "CTB": ["0"], "CTR": ["0"], "CV1": ["0"], "VRY": ["0", "1"],
    "SMG": ["0"], "PRS": ["0"], "WTF": ["0"],
}
SIGNALS_BY_STATE = {
    "air": ["SMG", "AC1", "VRY", "CV1", "GYR", "CTB", "CTR"],
    "water": ["SMG", "AC1", "VRY", "CV1", "GYR", "CTB", "CTR", "PRS", "WTF"],
}

# The supplied combined metadata contains legacy files only. For future raw
# analog.csv files, explicitly confirm electrical channel equivalents here.
# No old/new calibration or electrical equivalence is inferred from filenames.
NEW_RAW_COLUMNS = {
    "AC1": {"0": "ac1", "1": "ac2", "2": "ac3"},
    "GYR": {"0": "gy1", "1": "gy2", "2": "gy3"},
    "SMG": {"0": "mag"}, "PRS": {"0": "prs"}, "WTF": {"0": "wtf"},
    "CV1": {}, "VRY": {}, "CTB": {}, "CTR": {},
}

meta = pd.read_csv(META_PATH, dtype={"fault_id": str, "pump_id": str, "sample_id": str})
orig_ts = pd.read_csv(TIMESTAMPS_PATH, dtype=str)
meta["state"] = meta["state"].str.strip().str.lower()
orig_ts["state"] = orig_ts["state"].str.strip().str.lower()
for col in ["fault_id", "pump_id", "sample_id"]:
    meta[col] = meta[col].str.strip()
meta["sample_id_key"] = meta[["fault_id", "pump_id", "sample_id"]].agg("@".join, axis=1)
meta["pump_id_key"] = meta[["fault_id", "pump_id"]].agg("@".join, axis=1)
meta["state_id_key"] = meta["sample_id_key"] + "@" + meta["state"]
active_faults = meta["fault_id"].str.rstrip("_").map(lambda x: sum(c != "0" for c in x))
meta["fault_type"] = np.where(active_faults == 0, "nofault", np.where(active_faults == 1, "single", "multi"))
if not INCLUDE_MULTIFAULTS:
    meta = meta[meta["fault_type"] != "multi"].copy()
meta = meta.reset_index(drop=True)
if meta.empty or meta["state_id_key"].duplicated().any():
    raise ValueError("Metadata is empty or has duplicate sample/state keys.")
if not set(meta["state"]).issubset(STATES):
    raise ValueError("Unrecognized state in PUMP metadata.")
if orig_ts["id"].duplicated().any():
    raise ValueError("Duplicate signalwise changepoint IDs.")

orig_ts_lookup = {}
for row in orig_ts.itertuples(index=False):
    if not row.id.startswith(row.key + "_") or not row.id.endswith("@" + row.state):
        raise ValueError(f"Inconsistent signal/state in timestamp ID: {row.id}")
    cp = np.array(row.indices.split("#"), dtype=np.int64)
    if len(cp) < 2 or cp[0] < 0 or np.any(np.diff(cp) <= 0):
        raise ValueError(f"Invalid changepoints: {row.id}")
    orig_ts_lookup[row.id] = cp

# Pair Air/Water by explicit IDs, never by row position.
for sample_id, group in meta.groupby("sample_id_key", sort=False):
    if set(group["state"]) != set(STATES):
        raise ValueError(f"Missing Air/Water state: {sample_id}")
region_counts = {}
for index, row in meta.iterrows():
    for sig in SIGNALS_BY_STATE[row["state"]]:
        if sig not in meta or pd.isna(row[sig]) or not str(row[sig]).strip():
            raise ValueError(f"Missing {sig} path for {row['state_id_key']}")
        cp_key = f"{sig}_{row['state_id_key']}"
        if cp_key not in orig_ts_lookup:
            raise ValueError(f"Missing changepoints: {cp_key}")
        count = len(orig_ts_lookup[cp_key]) - 1
        previous = region_counts.setdefault(row["state"], count)
        if previous != count:
            raise ValueError(f"Inconsistent number of regions: {cp_key}")

print(f"{meta['sample_id_key'].nunique()} paired samples; {meta['fault_id'].nunique()} classes")
print("Regions per state:", region_counts)
meta.head()

315 paired samples; 18 classes
Regions per state: {'air': 2, 'water': 4}


,fault_id,pump_id,sample_id,state,AC1,GYR,CTB,CTR,CV1,VRY,PRS,SMG,WTF,sample_id_key,pump_id_key,state_id_key,fault_type
0,00000000000_,1,1,air,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,NaN,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,NaN,00000000000_@1@1,00000000000_@1,00000000000_@1@1@air,nofault
1,00000000000_,1,1,water,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,00000000000_@1@1,00000000000_@1,00000000000_@1@1@water,nofault
2,00000000000_,1,4,air,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,NaN,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,NaN,00000000000_@1@4,00000000000_@1,00000000000_@1@4@air,nofault
3,00000000000_,1,4,water,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,00000000000_@1@4,00000000000_@1,00000000000_@1@4@water,nofault
4,00000000000_,2,1,air,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,NaN,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,NaN,00000000000_@2@1,00000000000_@2,00000000000_@2@1@air,nofault


In [3]:
def resolve_path(value):
    path = str(value)
    for old_root, new_root in PATH_REPLACEMENTS.items():
        if path.startswith(old_root):
            path = str(new_root) + path[len(old_root):]
            break
    if os.name != "nt":
        path = path.replace("\\", "/")
    return path


def read_pump_signal(index, sig):
    """Read one signal without removing/reordering rows referenced by changepoints."""
    path = resolve_path(meta.loc[index, sig])
    header = pd.read_csv(path, nrows=0).columns.tolist()
    new_time = next((c for c in ("firmware_timestamp", "F_Timestamp") if c in header), None)
    timecol = new_time or "Timestamp"
    if timecol not in header:
        raise ValueError(f"Timestamp column missing: {path}")
    mapping = {}
    for col in SIGNAL_COLUMNS[sig]:
        # Prefer canonical signal names in merged files. Bare column_0/0
        # names apply only to separate legacy signal files.
        candidates = [f"{sig}_column_{col}"]
        if not new_time:
            candidates += [f"column_{col}", col]
        else:
            configured = NEW_RAW_COLUMNS.get(sig, {}).get(col)
            if configured:
                candidates.append(configured)
        found = next((c for c in candidates if c in header), None)
        if found is None:
            raise ValueError(f"Missing/unmapped {sig} column {col}: {path}. "
                             "For new raw electrical files, set NEW_RAW_COLUMNS explicitly.")
        mapping[found] = col
    orig = pl.read_csv(path, columns=[timecol] + list(mapping), n_threads=1,
                       low_memory=True, rechunk=False).rename({timecol: "Timestamp", **mapping})
    if new_time:
        orig = orig.with_columns((pl.col("Timestamp") / 1e6).alias("Timestamp"))
    times = orig["Timestamp"].to_numpy()
    valid_positions = np.flatnonzero(np.isfinite(times) & (times >= 0))
    if len(valid_positions) < 2:
        raise ValueError(f"Not enough valid timestamps: {path}")
    first, last = valid_positions[[0, -1]]
    duration = times[last] - times[first]
    if duration <= 0:
        raise ValueError(f"Invalid timestamp duration: {path}")
    fs = (last - first) / duration  # Legacy Timestamp is seconds; firmware time is microseconds.
    cp_key = f"{sig}_{meta.loc[index, 'state_id_key']}"
    cp = orig_ts_lookup[cp_key]
    if cp[-1] > orig.height:
        raise ValueError(f"Changepoints exceed raw-file row count: {cp_key}")
    return orig, fs, cp


def region_window(cp, region, fs, dist):
    """Fixed-size central window, with a margin inside both changepoints."""
    start, end = map(int, cp[region:region + 2])
    edge = int(np.ceil(REGION_EDGE_SECONDS * fs))
    available = end - start - 2 * edge
    if available < dist:
        raise ValueError(f"Region {region}: needs {dist} samples plus edge margins; "
                         f"only {available} usable samples. No crossing into another region.")
    ll = start + edge + (available - dist) // 2
    return ll, ll + dist


def finite_window(values, context):
    x = np.asarray(values, dtype=np.float64)
    if len(x) == 0 or not np.all(np.isfinite(x)):
        raise ValueError(f"Empty/non-finite feature window: {context}")
    return x


def feature_prefix(index, sig, region, col):
    return f"{sig}_{meta.loc[index, 'state']}_{region}_col_{col}"


def collect_features(extractor, description):
    """One output row per fault/pump/sample; state and region live in column names."""
    rows = []
    for sample_id, group in tqdm(meta.groupby("sample_id_key", sort=False),
                                 total=meta["sample_id_key"].nunique(), desc=description):
        row = {}
        for index in group.index:
            package = extractor(index)
            overlap = set(row).intersection(package)
            if overlap:
                raise ValueError(f"Duplicate state/region features: {overlap}")
            row.update(package)
        identity = group.iloc[0]
        for key in ["sample_id_key", "fault_id", "pump_id", "sample_id", "pump_id_key", "fault_type"]:
            row[key] = identity[key]
        rows.append(row)
        gc.collect()
    result = pd.DataFrame(rows)
    if result.empty or result.isna().any().any():
        raise ValueError("Empty/incomplete feature table; inspect sample regions and signals.")
    numeric = result.select_dtypes(include=np.number)
    if not np.isfinite(numeric.to_numpy()).all():
        raise ValueError("Non-finite feature values; inspect the raw signals.")
    return result

In [4]:
def get_amplitude(signal, prominence=0.15, distance=600, plot=False):
    signal = np.asarray(signal)
    peak_to_peak_amplitude = []
    peaks, _ = find_peaks(signal, prominence=prominence, distance=distance)
    troughs, _ = find_peaks(-signal, prominence=prominence, distance=distance)
    if plot:
        time = np.linspace(0, 10, signal.shape[0])
        plt.figure(figsize=(10, 4))
        plt.plot(time, signal, label='Signal')
        plt.plot(time[peaks], signal[peaks], 'rx', label='Peaks')
        plt.plot(time[troughs], signal[troughs], 'bx', label='Troughs')
        plt.xlabel('Time')
        plt.ylabel('Amplitude')
        plt.title(f'Signal with Peak-to-Peak Amplitude Points {signal.shape[0]}')
        plt.legend()
        plt.grid(True)
        plt.show()
    ele_len = min(len(peaks), len(troughs))
    peaks = peaks[:ele_len]
    troughs = troughs[:ele_len]
    peak_to_peak_amplitude = np.abs(signal[peaks] - signal[troughs])
    valid_mask = peak_to_peak_amplitude < 10000
    peak_to_peak_amplitude = peak_to_peak_amplitude[valid_mask]
    peaks = peaks[valid_mask]
    troughs = troughs[valid_mask]
    return (peak_to_peak_amplitude, peaks, troughs)

In [5]:
PHYSICS_SIGNALS = ["CV1", "CTB", "CTR", "VRY", "SMG"]
TRANSIENT_SECONDS = 0.62  # Same short observation window as FAN.
PEAK_MIN_DISTANCE_SECONDS = 0.01  # 500 samples at the FAN deployed 50 kHz rate.
STABILITY_TOLERANCE = 0.10
CONSECUTIVE_CYCLES = 5

def get_peaks_troughs(x_subset, fs, _plot=False):
    p2p, peaks, troughs = get_amplitude(x_subset,
        distance=max(1, int(round(fs * PEAK_MIN_DISTANCE_SECONDS))), plot=_plot)
    if len(p2p) < CONSECUTIVE_CYCLES:
        raise ValueError("Too few electrical waveform cycles for a stability estimate")
    steady_amp = np.median(p2p[-min(5, len(p2p)):])
    relative_error = np.abs(p2p - steady_amp) / (steady_amp + 1e-8)
    stable_index = None
    for i in range(len(relative_error) - CONSECUTIVE_CYCLES + 1):
        if np.all(relative_error[i:i + CONSECUTIVE_CYCLES] <= STABILITY_TOLERANCE):
            stable_index = int(peaks[i])
            break
    # If no stable block is found, observation length is a lower bound, not
    # a falsely reported settling time. Keep the flag alongside the value.
    observed = stable_index is not None
    return {"cycle_count": float(max(len(peaks), len(troughs))),
            "stability_observed": float(observed),
            "settling_time_s": stable_index / fs if observed else len(x_subset) / fs}


def extract_physics_features(index):
    feats = {}
    for sig in PHYSICS_SIGNALS:
        orig, fs, cp = read_pump_signal(index, sig)
        for region in range(len(cp) - 1):
            ll = int(cp[region])
            dist = int(round(fs * TRANSIENT_SECONDS))
            if dist < 2 or ll + dist > cp[region + 1]:
                raise ValueError(f"Too short transient window: {meta.loc[index, 'state_id_key']}/{sig}/{region}")
            for col in SIGNAL_COLUMNS[sig]:
                key = feature_prefix(index, sig, region, col)
                x = finite_window(orig[col].slice(ll, dist).to_numpy(), key)
                x = x - np.mean(x)
                try:
                    values = get_peaks_troughs(x, fs)
                except ValueError as error:
                    raise ValueError(f"{key}: {error}") from error
                feats.update({f"{key}_{name}": value for name, value in values.items()})
        del orig
    return feats

In [6]:
physics_feats = collect_features(extract_physics_features, "PUMP physics")
physics_feats.head()

PUMP physics: 100%|██████████| 315/315 [19:59<00:00,  3.81s/it]


,CV1_air_0_col_0_cycle_count,CV1_air_0_col_0_stability_observed,CV1_air_0_col_0_settling_time_s,CV1_air_1_col_0_cycle_count,CV1_air_1_col_0_stability_observed,CV1_air_1_col_0_settling_time_s,CTB_air_0_col_0_cycle_count,CTB_air_0_col_0_stability_observed,CTB_air_0_col_0_settling_time_s,CTB_air_1_col_0_cycle_count,...,SMG_water_2_col_0_settling_time_s,SMG_water_3_col_0_cycle_count,SMG_water_3_col_0_stability_observed,SMG_water_3_col_0_settling_time_s,sample_id_key,fault_id,pump_id,sample_id,pump_id_key,fault_type
0,43.0,0.0,0.620011,45.0,0.0,0.620011,46.0,0.0,0.620003,43.0,...,0.619990,45.0,1.0,0.247730,00000000000_@1@1,00000000000_,1,1,00000000000_@1,nofault
1,51.0,1.0,0.439981,48.0,0.0,0.620002,44.0,0.0,0.620003,44.0,...,0.620010,45.0,1.0,0.281565,00000000000_@1@4,00000000000_,1,4,00000000000_@1,nofault
2,50.0,0.0,0.619999,47.0,0.0,0.619999,45.0,0.0,0.619999,44.0,...,0.619998,45.0,0.0,0.619998,00000000000_@2@1,00000000000_,2,1,00000000000_@2,nofault
3,49.0,0.0,0.619994,47.0,0.0,0.619994,46.0,0.0,0.620003,47.0,...,0.619998,43.0,0.0,0.619998,00000000000_@2@2,00000000000_,2,2,00000000000_@2,nofault
4,46.0,0.0,0.619998,44.0,0.0,0.619998,46.0,0.0,0.620003,44.0,...,0.466150,46.0,1.0,0.262565,00000000000_@2@3,00000000000_,2,3,00000000000_@2,nofault


In [7]:
physics_feats.to_csv(OUTPUT_DIR / "PUMP_physics_features.csv", index=False)
print("Saved", physics_feats.shape)

Saved (315, 114)
